# Periodic committor: CPU preflight, then GPU pilot

Do not spend GPU compute on the full pilot until the preflight has succeeded.

1. Select a T4 GPU in **Runtime → Change runtime type**. The preflight itself runs on CPU; the T4 is needed only for the later pilot.
2. Run the first two code cells. The first installs and tests the package; the second runs a complete two-epoch CPU comparison, including finite-difference reference, three methods, checkpoints, summary, and figures. It should take only a short time and uses no GPU computation.
3. Inspect the printed preflight folder and its figures. If it completed, run the third code cell to start the actual GPU pilot: three methods, 6,000 epochs each, and 128/256-grid references.
4. Run the display cell after the GPU pilot finishes. Use the final cell to resume an interrupted GPU pilot.

In a fresh Colab runtime, the setup cell asks once for a GitHub token. Paste the repository-read token. It only lets Colab clone this private repository; it is not used for training and is not saved.

In [ ]:
from pathlib import Path
from getpass import getpass
import base64
import shutil
import subprocess

from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/boundary-adapted-pinns')
REPO_URL = 'https://github.com/n2pakbong/boundary-adapted-pinns.git'
BRANCH = 'committor-colab'

if not (REPO / 'pyproject.toml').is_file():
    if REPO.exists():
        shutil.rmtree(REPO)  # only a failed partial clone at this fixed Colab path
    token = getpass('Paste your GitHub token, then press Enter: ')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    try:
        subprocess.run(
            ['git', '-c', f'http.extraheader=AUTHORIZATION: basic {auth}',
             'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)],
            check=True,
        )
    finally:
        del token, auth
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', REPO_URL], check=True)
else:
    print('Using the existing clone at', REPO)

%cd /content/boundary-adapted-pinns
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
subprocess.run(['python', '-m', 'pytest', '-q'], check=True)
print('Setup and unit tests passed.')

In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess

preflight_name = 'preflight_' + datetime.now().strftime('%Y%m%d_%H%M%S')
PREFLIGHT_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{preflight_name}'
Path(PREFLIGHT_ROOT).mkdir(parents=True, exist_ok=False)
print('CPU preflight results will be saved in:', PREFLIGHT_ROOT)

subprocess.run([
    'met-pinn-periodic-comparison',
    'configs/periodic_committor/preflight_comparison_cpu.json',
    '--output-dir', PREFLIGHT_ROOT,
], check=True)
print('CPU preflight completed. Inspect summary.json and figures before starting the GPU pilot.')

In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError('The full pilot needs a GPU. Select a T4 GPU under Runtime → Change runtime type, then rerun setup and the CPU preflight.')

run_name = 'periodic_committor_' + datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{run_name}'
Path(RUN_ROOT).mkdir(parents=True, exist_ok=False)
print('GPU pilot results will be saved in:', RUN_ROOT)

subprocess.run([
    'met-pinn-periodic-comparison',
    'configs/periodic_committor/overnight_comparison.json',
    '--output-dir', RUN_ROOT,
], check=True)

In [ ]:
import json
from IPython.display import Image, display

with open(f'{RUN_ROOT}/summary.json') as handle:
    print(json.dumps(json.load(handle), indent=2))
for filename in ['losses.png', 'committor_boundary_adapted.png', 'committor_naive_product.png', 'committor_penalty.png']:
    display(Image(filename=f'{RUN_ROOT}/{filename}'))

## Resume an interrupted GPU pilot

Rerun the setup cell in a fresh runtime. Then replace the quoted folder below with the `RUN_ROOT` printed by the interrupted GPU pilot and execute the cell. Each method resumes from its own `last.pt` checkpoint.

In [ ]:
# RUN_ROOT = '/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/periodic_committor_YYYYMMDD_HHMMSS'
# subprocess.run([
#     'met-pinn-periodic-comparison',
#     'configs/periodic_committor/overnight_comparison.json',
#     '--output-dir', RUN_ROOT,
#     '--resume',
# ], check=True)